<a href="https://colab.research.google.com/github/boulefradas-rgb/room-assignment-optimization/blob/main/Final_Program.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SECTION 1: IMPORTS AND INITIAL SETUP

In [ ]:
import pulp
from collections import defaultdict
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("Starting optimizer...")

Starting optimizer...


# SECTION 2: DATA DEFINITION

In [ ]:
TIME_LABELS = {
    1:"08:00-09:30", 2:"09:30-11:00", 3:"11:00-12:30",
    4:"12:30-14:00", 5:"14:00-15:30", 6:"15:30-17:00",
}
T = list(TIME_LABELS.keys())

DAY_LABELS = {
    1:"Saturday", 2:"Sunday",    3:"Monday",
    4:"Tuesday",  5:"Wednesday", 6:"Thursday",
}
D = list(DAY_LABELS.keys())

ROOMS = {
    "Salle10":40, "Salle11":40,
    "Salle15":70, "Salle16":70, "AmphiA":120,
}
R = list(ROOMS.keys())

LECTURES_RAW = [
    #Saturday
    ("S01","Physique1",   42,1,1),("S02","Chimie1",      42,1,1),("S03","Geometrie",   42,1,1),
    ("S04","Chimie1",     42,1,2),("S05","Physique1",    21,1,2),("S06","Geometrie",   21,1,2),
    ("S07","Analyse2",    21,1,2),("S08","Chimie1",     105,1,3),("S09","Analyse2",    42,1,3),
    ("S10","Didactique1", 30,1,3),("S11","Analyse2",    21,1,4),("S12","Didactique1", 30,1,4),
    ("S13","Physique1",  105,1,5),("S14","Chimie1",     21,1,6),("S15","Physique1",   42,1,6),
    #Sunday
    ("U01","Sci_Education",10,2,1),("U02","MathBase",   42,2,1),("U03","Analyse1",    21,2,1),
    ("U04","Topologie",   42,2,1),("U05","Mesure_Int1", 30,2,1),("U06","Analyse1",   105,2,2),
    ("U07","Informatique",42,2,2),("U08","Mesure_Int1", 30,2,2),("U09","Geo_Diff",   30,2,2),
    ("U10","Analyse1",    42,2,3),("U11","Algebre1",    21,2,3),("U12","Topologie",  21,2,3),
    ("U13","Probabilité2",30,2,3),("U14","Algebre1",    42,2,4),("U15","Proba_Stat1",30,2,4),
    ("U16","Topologie",  21,2,5),("U17","Algebre1",105,2,5),("U18","Mesure et Int2",30,2,5),
    ("U19","Mesure et Int2",30,2,6),
    #Monday
    ("M01","Analyse1",   105,3,1),("M02","Analyse_Num1",30,3,1),("M03","Algebre4",   30,3,1),
    ("M04","Mecanique",   37,3,1),("M05","Math_Base",   42,3,2),("M06","Algebre_Gén",21,3,2),
    ("M07","Algebre4",    30,3,2),("M08","Mecanique",   37,3,2),("M09","Tic",        105,3,3),
    ("M10","Algebre_Gén", 21,3,3),("M11","Calcul_Diff", 21,3,3),("M12","Analyse_Comp2",30,3,3),
    ("M13","Didactique2", 37,3,3),("M14","Calcul_Diff", 42,3,5),("M15","Psycho3",    30,3,5),
    ("M16","Analyse_Comp2",30,3,5),("M17","Analyse_Num2",37,3,5),("M18","Psycho2",   42,3,6),
    ("M19","Didactique2", 37,3,6),
    #Tuesday
    ("T01","Analyse1",    42,4,1),("T02","Topologie",   42,4,1),("T03","Analyse_Comp1",30,4,1),
    ("T04","Analyse_Num2",37,4,1),("T05","Analyse1",   105,4,2),("T06","Algebre_Gén", 42,4,2),
    ("T07","Analyse_Comp1",30,4,2),("T08","Legislation",64,4,2),("T09","Analyse1",    21,4,3),
    ("T10","Topologie",   21,4,3),("T11","Geometrie",   21,4,3),("T12","Geo_Diff",   30,4,3),
    ("T13","Algebre1",    42,4,4),("T14","Logique",     30,4,4),("T15","Evaluation",  30,4,4),
    ("T16","Histoire_maths",37,4,4),("T17","Algebre1", 105,4,5),("T18","Algebre_lin", 21,4,5),
    ("T19","Topologie",   21,4,5),("T20","Histoire_maths",37,4,5),
    #wednesday
    ("W01","Analyse1",    42,5,1),("W02","Algebre_lin", 42,5,1),("W03","Algebre3",   30,5,1),
    ("W04","Analyse1",    42,5,2),("W05","Calcul_Diff", 21,5,2),("W06","Algebre_lin",21,5,2),
    ("W07","Algebre3",    30,5,2),("W08","Probabilité2", 30,5,2),("W09","Histoire_Sci1",85,5,3),
     ("W10","Proba_Stat1", 30,5,3),("W11","Equa_Diff",  30,5,3),("W12","Anglais",    105,5,5),
    ("W13","Geo_affine",  30,5,5),("W14","Equa_Diff",  30,5,5),("W15","Geo_affine",  30,5,6),
    #Thursday
    ("H01","Analyse_Num1",30,6,1),("H02","Analyse_Fonc",37,6,1),("H03","Analyse_Num1",30,6,2),
    ("H04","Analyse_Fonc",37,6,2),("H05","RO",  37,6,3),("H06","RO",         37,6,4),
]

lectures = {lid:{"course":c,"students":s,"day":d,"slot":t} for lid,c,s,d,t in LECTURES_RAW}
L = list(lectures.keys())

ORIGINAL_ASSIGNMENTS = {
    #Saturday
    "S01":"Salle16","S02":"Salle15","S03":"Salle15","S04":"Salle16","S05":"Salle16",
    "S06":"Salle10","S07":"Salle11","S08":"AmphiA", "S09":"Salle16","S10":"Salle15",
    "S11":"Salle16","S12":"Salle15","S13":"AmphiA", "S14":"Salle16","S15":"Salle15",
    #Sunday
    "U01":"Salle10","U02":"Salle15","U03":"Salle16","U04":"Salle16","U05":"Salle10",
    "U06":"AmphiA", "U07":"Salle15","U08":"Salle10","U09":"Salle15","U10":"Salle16",
    "U11":"Salle15","U12":"Salle10","U13":"Salle11","U14":"Salle15","U15":"Salle10",
    "U16":"Salle11","U17":"AmphiA","U18":"Salle11","U19":"Salle11",
    #Monday
    "M01":"AmphiA", "M02":"Salle15","M03":"Salle11","M04":"Salle16","M05":"Salle16",
    "M06":"Salle11","M07":"Salle11","M08":"Salle10","M09":"AmphiA", "M10":"Salle11",
    "M11":"Salle10","M12":"Salle15","M13":"Salle16","M14":"Salle15","M15":"AmphiA",
    "M16":"Salle10","M17":"Salle11","M18":"AmphiA", "M19":"Salle11",
    #Tuesday
    "T01":"Salle15","T02":"Salle16","T03":"Salle10","T04":"Salle11","T05":"AmphiA",
    "T06":"Salle15","T07":"Salle10","T08":"Salle16","T09":"Salle16","T10":"Salle11",
    "T11":"Salle10","T12":"Salle15","T13":"Salle15","T14":"Salle10","T15":"Salle16",
    "T16":"Salle11","T17":"AmphiA", "T18":"Salle11","T19":"Salle10","T20":"Salle11",
    #wednesday
    "W01":"Salle16","W02":"Salle15","W03":"Salle10","W04":"Salle16","W05":"Salle11",
    "W06":"Salle15","W07":"Salle10","W08":"Salle15","W09":"AmphiA", "W10":"Salle10",
    "W11":"Salle11","W12":"AmphiA", "W13":"Salle10","W14":"Salle15","W15":"Salle15",
    #Thursday
    "H01":"Salle11","H02":"Salle16","H03":"Salle10","H04":"Salle16","H05":"Salle11","H06":"Salle16",
}

ALPHA=0.10; LAMBDA=0.1; BETA=0.5; MU=0.4; BETA2=0.5

# SECTION 3: HELPER FUNCTIONS

In [ ]:
def seated(l, r):
    s=lectures[l]["students"]; c=ROOMS[r]
    return s if c>=s else c

def build_A():
    A={}
    for l in L:
        A[l]={d:{t:0 for t in T} for d in D}
        A[l][lectures[l]["day"]][lectures[l]["slot"]]=1
    return A

def count_transitions(asgn):
    total=0
    for d in D:
        grid={r:{t:0 for t in T} for r in R}
        for l,r in asgn.items():
            if lectures[l]["day"]==d: grid[r][lectures[l]["slot"]]=1
        for r in R:
            seq=[grid[r][t] for t in T]
            total+=sum(1 for i in range(1,len(T)) if seq[i]!=seq[i-1])
    return total

A=build_A()

slot_usage=defaultdict(list)
for l,r in ORIGINAL_ASSIGNMENTS.items():
    slot_usage[(r,lectures[l]["day"],lectures[l]["slot"])].append(l)
lost_room=set()
for lids in slot_usage.values():
    for l in lids[1:]: lost_room.add(l)

total_enrolled=sum(lectures[l]["students"] for l in L)
total_seated_orig=0
overbooked_orig=[]
for l in L:
    r=ORIGINAL_ASSIGNMENTS.get(l,"NONE"); s=lectures[l]["students"]
    if l in lost_room: seat=0; overbooked_orig.append(l)
    else:
        seat=seated(l,r) if r in ROOMS else 0
        if seat<s: overbooked_orig.append(l)
    total_seated_orig+=seat
orig_transitions=count_transitions(ORIGINAL_ASSIGNMENTS)

# SECTION 4: STAGE 1 — Maximize seated students

In [ ]:
print("Solving First stage...")
stage1=pulp.LpProblem("FirstStage",pulp.LpMaximize)
x=pulp.LpVariable.dicts("x",[(l,r) for l in L for r in R],cat="Binary")
stage1+=(
    pulp.lpSum(seated(l,r)*x[(l,r)]*A[l][lectures[l]["day"]][lectures[l]["slot"]] for l in L for r in R)
    -LAMBDA*pulp.lpSum(max(0,BETA*ROOMS[r]-lectures[l]["students"])*x[(l,r)] for l in L for r in R),
    "Obj1")
for l in L: stage1+=(pulp.lpSum(x[(l,r)] for r in R)==1,f"C8_{l}")
for r in R:
    for d in D:
        for t in T: stage1+=(pulp.lpSum(x[(l,r)]*A[l][d][t] for l in L)<=1,f"C9_{r}_{d}_{t}")
for l in L:
    for r in R: stage1+=(ROOMS[r]>=x[(l,r)]*lectures[l]["students"]*(1-ALPHA),f"C10_{l}_{r}")
stage1.solve(pulp.PULP_CBC_CMD(msg=0))
BEST=int(round(pulp.value(pulp.lpSum(
    seated(l,r)*x[(l,r)]*A[l][lectures[l]["day"]][lectures[l]["slot"]] for l in L for r in R))))
asgn1={l:r for l in L for r in R if pulp.value(x[(l,r)]) and pulp.value(x[(l,r)])>0.5}
total_s1=sum(seated(l,asgn1[l]) for l in L)
ob_s1=len([l for l in L if seated(l,asgn1.get(l,""))<lectures[l]["students"]])
trans_s1=count_transitions(asgn1)
print(f"  Done — BEST={BEST}, transitions={trans_s1}")

Solving First stage...
  Done — BEST=3729, transitions=67


# SECTION 5: STAGE 2 — Minimize transitions

In [ ]:
print("Solving Second stage...")
stage2=pulp.LpProblem("SecondStage",pulp.LpMinimize)
x2=pulp.LpVariable.dicts("x2",[(l,r) for l in L for r in R],cat="Binary")
occ=pulp.LpVariable.dicts("occ",[(r,d,t) for r in R for d in D for t in T],cat="Binary")
c=pulp.LpVariable.dicts("c",[(l,d,t,r) for l in L for d in D for t in T for r in R],cat="Binary")
stage2+=(
    pulp.lpSum(c[(l,d,t,r)] for l in L for d in D for t in T for r in R)
    +MU*pulp.lpSum(max(0,BETA2*ROOMS[r]-lectures[l]["students"])*x2[(l,r)] for l in L for r in R),
    "Obj2")
for l in L: stage2+=(pulp.lpSum(x2[(l,r)] for r in R)==1,f"C8_{l}")
for r in R:
    for d in D:
        for t in T: stage2+=(pulp.lpSum(x2[(l,r)]*A[l][d][t] for l in L)<=1,f"C9_{r}_{d}_{t}")
for l in L:
    for r in R: stage2+=(ROOMS[r]>=x2[(l,r)]*lectures[l]["students"]*(1-ALPHA),f"C10_{l}_{r}")
stage2+=(pulp.lpSum(
    seated(l,r)*x2[(l,r)]*A[l][lectures[l]["day"]][lectures[l]["slot"]] for l in L for r in R)==BEST,"C11")
for r in R:
    for d in D:
        for t in T:
            lhs=pulp.lpSum(x2[(l,r)]*A[l][d][t] for l in L)
            stage2+=(occ[(r,d,t)]<=lhs,f"OUB_{r}_{d}_{t}")
            stage2+=(occ[(r,d,t)]>=lhs,f"OLB_{r}_{d}_{t}")
for r in R:
    for d in D:
        for t in T:
            if t>=2:
                for l in L:
                    stage2+=(c[(l,d,t,r)]>=occ[(r,d,t)]-occ[(r,d,t-1)],f"CR_{l}_{d}_{t}_{r}")
                    stage2+=(c[(l,d,t,r)]>=occ[(r,d,t-1)]-occ[(r,d,t)],f"CF_{l}_{d}_{t}_{r}")
            else:
                for l in L: stage2+=(c[(l,d,t,r)]==0,f"CZ_{l}_{d}_{t}_{r}")
stage2.solve(pulp.PULP_CBC_CMD(msg=0))
asgn2={l:r for l in L for r in R if pulp.value(x2[(l,r)]) and pulp.value(x2[(l,r)])>0.5}
total_s2=sum(seated(l,asgn2[l]) for l in L)
ob_s2=len([l for l in L if seated(l,asgn2.get(l,""))<lectures[l]["students"]])
trans_s2=count_transitions(asgn2)
print(f"  Done — transitions={trans_s2}")

Solving Second stage...
  Done — transitions=41


# SECTION 6: EXCEL HELPERS

In [ ]:
def brd():
    s=Side(style="thin")
    return Border(left=s,right=s,top=s,bottom=s)

def fl(hex6):
    return PatternFill("solid",start_color=hex6,fgColor=hex6)

def hfont(size=11,bold=True):
    return Font(bold=bold,color="FFFFFF",name="Arial",size=size)

def dfont(size=10,bold=False,color="000000"):
    return Font(bold=bold,color=color,name="Arial",size=size)

def center():
    return Alignment(horizontal="center",vertical="center")

def wrap():
    return Alignment(horizontal="center",vertical="center",wrap_text=True)

PAL={
    "orig": {"hdr":"C55A11","light":"FCE4D6","mid":"F4B183"},
    "s1":   {"hdr":"1F4E79","light":"DEEAF1","mid":"9DC3E6"},
    "s2":   {"hdr":"1E6B3C","light":"E2EFDA","mid":"A9D18E"},
    "cmp":  {"hdr":"4472C4","light":"DAE3F3","mid":"B4C7E7"},
}

# ── Status labels (matching thesis terminology exactly) ──────
STATUS_FEASIBLE          = "Feasible"           # constraint satisfied
STATUS_CONFLICT          = "Conflict"           # violates constraint (9)
STATUS_CAPACITY_VIOLATION= "Capacity violation" # violates constraint (10)


In [ ]:
# ============================================================
# SHEET 1: ASSIGNMENT TABLE
# ============================================================
def make_assignment_sheet(ws, asgn, pal, label, is_orig=False):
    ws.merge_cells("A1:J1")
    ws["A1"]=label
    ws["A1"].font=Font(bold=True,color="FFFFFF",name="Arial",size=13)
    ws["A1"].fill=fl(pal["hdr"]); ws["A1"].alignment=center()
    ws.row_dimensions[1].height=26

    hdrs=["ID","Course","Day","Enrolled students","Time Slot",
          "Room","Capacity","Seated students","Unseated","Status"]
    for col,h in enumerate(hdrs,1):
        cell=ws.cell(row=2,column=col,value=h)
        cell.font=hfont(); cell.fill=fl(pal["hdr"])
        cell.alignment=center(); cell.border=brd()
    ws.row_dimensions[2].height=18

    row=3
    for l in L:
        info=lectures[l]; r=asgn.get(l,"NONE"); s=info["students"]; cap=ROOMS.get(r,0)
        if is_orig and l in lost_room:
            seat=0
            status=STATUS_CONFLICT
            r_show=f"{r}(!)"
        else:
            seat=seated(l,r) if r in ROOMS else 0
            if seat<s:
                status=STATUS_CAPACITY_VIOLATION
            else:
                status=STATUS_FEASIBLE
            r_show=r

        bg="FFE0E0" if status in (STATUS_CONFLICT, STATUS_CAPACITY_VIOLATION) \
            else (pal["light"] if row%2==0 else "FFFFFF")

        vals=[l,info["course"],DAY_LABELS[info["day"]],s,
              TIME_LABELS[info["slot"]],r_show,cap,seat,s-seat,status]
        for col,val in enumerate(vals,1):
            cell=ws.cell(row=row,column=col,value=val)
            cell.fill=fl(bg); cell.alignment=center(); cell.border=brd()
            cell.font=dfont(color="CC0000",bold=True) \
                if (col==10 and status in (STATUS_CONFLICT, STATUS_CAPACITY_VIOLATION)) \
                else dfont()
        row+=1

    t_e=total_enrolled
    t_s=total_seated_orig if is_orig else (total_s1 if "First stage" in label else total_s2)
    t_u=t_e-t_s; pct=round(100*t_s/t_e,1) if t_e else 0
    ws.cell(row=row,column=1,value="TOTAL")
    ws.cell(row=row,column=4,value=t_e)
    ws.cell(row=row,column=8,value=t_s)
    ws.cell(row=row,column=9,value=t_u)
    for col in range(1,11):
        cell=ws.cell(row=row,column=col)
        cell.fill=fl(pal["hdr"]); cell.border=brd()
        cell.font=hfont(); cell.alignment=center()
    row+=2

    ws.cell(row=row,column=1,value="STATISTICS").font=Font(
        bold=True,name="Arial",size=11,color=pal["hdr"])
    row+=1
    if is_orig:
        ob_count=len([l for l in overbooked_orig if l not in lost_room])
        stats=[
            ("Enrolled students",          t_e),
            ("Seated students",            f"{t_s}  ({pct}%)"),
            ("Unseated students",          t_u),
            ("Sessions with conflict",     len(lost_room)),
            ("Capacity violations",        ob_count),
            ("Transitions",                orig_transitions),
        ]
    else:
        ob  = ob_s1  if "First stage"  in label else ob_s2
        tr  = trans_s1 if "First stage" in label else trans_s2
        par = f"λ={LAMBDA}, β={BETA}" if "First stage" in label else f"μ={MU}, β={BETA2}"
        stats=[
            ("Enrolled students",  t_e),
            ("Seated students",    f"{t_s}  ({pct}%)"),
            ("Unseated students",  t_u),
            ("Capacity violations",ob),
            ("Transitions",        tr),
            ("ILP Parameters",     par),
        ]
    for lbl,val in stats:
        ws.cell(row=row,column=1,value=lbl).font=dfont(bold=True)
        ws.cell(row=row,column=2,value=val).font=dfont()
        ws.cell(row=row,column=1).fill=fl(pal["light"]); ws.cell(row=row,column=1).border=brd()
        ws.cell(row=row,column=2).fill=fl(pal["light"]); ws.cell(row=row,column=2).border=brd()
        row+=1

    for i,w in enumerate([7,18,11,12,14,12,9,12,9,18],1):
        ws.column_dimensions[get_column_letter(i)].width=w


In [ ]:
# ============================================================
# SHEET 2: ROOM TIMETABLE — room → days layout
# ============================================================
def make_room_sheet_by_room(ws, asgn, pal, label, is_orig=False):
    n_cols=1+len(T)
    ws.merge_cells(start_row=1,start_column=1,end_row=1,end_column=n_cols)
    ws["A1"]=f"ROOM TIMETABLES — {label}"
    ws["A1"].font =Font(bold=True,color="FFFFFF",name="Arial",size=13)
    ws["A1"].fill =fl(pal["hdr"])
    ws["A1"].alignment=center()
    ws.row_dimensions[1].height=26

    grid=defaultdict(lambda: None)
    for l,r in asgn.items():
        grid[(r,lectures[l]["day"],lectures[l]["slot"])]=l

    current_row=2

    for r in R:
        cap=ROOMS[r]

        # Room header
        ws.merge_cells(start_row=current_row,start_column=1,
                       end_row=current_row,end_column=n_cols)
        cell=ws.cell(row=current_row,column=1,
                     value=f"  {r}   (Capacity: {cap})")
        cell.font     =Font(bold=True,color="FFFFFF",name="Arial",size=12)
        cell.fill     =fl(pal["mid"])
        cell.alignment=Alignment(horizontal="left",vertical="center")
        cell.border   =brd()
        ws.row_dimensions[current_row].height=22
        current_row+=1

        # Column headers
        ws.cell(row=current_row,column=1,value="Day").font     =hfont(size=10)
        ws.cell(row=current_row,column=1).fill     =fl(pal["hdr"])
        ws.cell(row=current_row,column=1).alignment=center()
        ws.cell(row=current_row,column=1).border   =brd()
        for col_idx,t in enumerate(T,2):
            cell=ws.cell(row=current_row,column=col_idx,value=TIME_LABELS[t])
            cell.font=hfont(size=10); cell.fill=fl(pal["hdr"])
            cell.alignment=center(); cell.border=brd()
        ws.row_dimensions[current_row].height=18
        current_row+=1

        # One row per day
        for di,d in enumerate(D):
            bg_day=pal["light"] if di%2==0 else "FFFFFF"
            cell=ws.cell(row=current_row,column=1,value=DAY_LABELS[d])
            cell.font=dfont(bold=True); cell.fill=fl(bg_day)
            cell.alignment=center(); cell.border=brd()

            for col_idx,t in enumerate(T,2):
                lec=grid[(r,d,t)]
                if lec:
                    s   =lectures[lec]["students"]
                    seat=seated(lec,r)
                    ok  =seat>=s
                    if is_orig and lec in lost_room:
                        txt =f"{lectures[lec]['course']}\n({s} st.) Conflict"
                        bg  ="FFD7D7"; fcol="CC0000"
                    elif not ok:
                        txt =f"{lectures[lec]['course']}\n({s}→{seat}) Cap.viol."
                        bg  ="FFD7D7"; fcol="CC0000"
                    else:
                        txt =f"{lectures[lec]['course']}\n({s}→{seat})"
                        bg  ="E2EFDA"; fcol="000000"
                    cell=ws.cell(row=current_row,column=col_idx,value=txt)
                    cell.fill=fl(bg); cell.font=dfont(size=9,color=fcol)
                    cell.alignment=Alignment(horizontal="center",
                                             vertical="center",wrap_text=True)
                    cell.border=brd()
                else:
                    cell=ws.cell(row=current_row,column=col_idx,value="—")
                    cell.fill=fl("F5F5F5"); cell.font=dfont(size=9,color="AAAAAA")
                    cell.alignment=center(); cell.border=brd()

            ws.row_dimensions[current_row].height=32
            current_row+=1

        current_row+=1

    ws.column_dimensions["A"].width=13
    for col_idx in range(2,2+len(T)):
        ws.column_dimensions[get_column_letter(col_idx)].width=20

In [ ]:
# ============================================================
# BUILD WORKBOOK
# ============================================================
print("Building Excel file...")
wb=Workbook()
wb.remove(wb.active)

# 1. Original assignment
ws=wb.create_sheet("Original — Assignment")
make_assignment_sheet(ws,ORIGINAL_ASSIGNMENTS,PAL["orig"],
                      "ORIGINAL HANDMADE ASSIGNMENT TABLE",is_orig=True)

# 2. Original room timetable
ws=wb.create_sheet("Original — Room Timetable")
make_room_sheet_by_room(ws,ORIGINAL_ASSIGNMENTS,PAL["orig"],
                        "ORIGINAL",is_orig=True)

# 3. First stage assignment
ws=wb.create_sheet("First stage — Assignment")
make_assignment_sheet(ws,asgn1,PAL["s1"],
                      "FIRST STAGE — MAXIMIZE SEATED STUDENTS")

# 4. First stage room timetable
ws=wb.create_sheet("First stage — Room Timetable")
make_room_sheet_by_room(ws,asgn1,PAL["s1"],"FIRST STAGE")

# 5. Second stage assignment
ws=wb.create_sheet("Second stage — Assignment")
make_assignment_sheet(ws,asgn2,PAL["s2"],
                      "SECOND STAGE — MINIMIZE TRANSITIONS")

# 6. Second stage room timetable
ws=wb.create_sheet("Second stage — Room Timetable")
make_room_sheet_by_room(ws,asgn2,PAL["s2"],"SECOND STAGE")

# 7. Final comparison
ws_cmp=wb.create_sheet("Final Comparison")
pal=PAL["cmp"]
ws_cmp.merge_cells("A1:F1")
ws_cmp["A1"]="FINAL OPTIMIZATION COMPARISON SUMMARY"
ws_cmp["A1"].font=Font(bold=True,color="FFFFFF",name="Arial",size=14)
ws_cmp["A1"].fill=fl(pal["hdr"]); ws_cmp["A1"].alignment=center()
ws_cmp.row_dimensions[1].height=30

cmp_hdrs=["Metric","Original\n(Handmade)",
          "First stage\n(Max Seated)",
          "Second stage\n(Min Transitions)",
          "1st vs Original","2nd vs Original"]
for col,h in enumerate(cmp_hdrs,1):
    cell=ws_cmp.cell(row=2,column=col,value=h)
    cell.font=hfont(size=11); cell.fill=fl(pal["hdr"])
    cell.alignment=wrap(); cell.border=brd()
ws_cmp.row_dimensions[2].height=40

pct_o =round(100*total_seated_orig/total_enrolled,1)
pct_s1=round(100*total_s1/total_enrolled,1)
pct_s2=round(100*total_s2/total_enrolled,1)
ob_orig_count=len(overbooked_orig)

cmp_data=[
    ("Enrolled students",
     total_enrolled,total_enrolled,total_enrolled,"—","—","DEEAF1"),
    ("Seated students",
     f"{total_seated_orig} ({pct_o}%)",
     f"{total_s1} ({pct_s1}%)",
     f"{total_s2} ({pct_s2}%)",
     f"+{total_s1-total_seated_orig}",
     f"+{total_s2-total_seated_orig}","E2EFDA"),
    ("Unseated students",
     total_enrolled-total_seated_orig,
     total_enrolled-total_s1,
     total_enrolled-total_s2,
     f"-{total_s1-total_seated_orig}",
     f"-{total_s2-total_seated_orig}","E2EFDA"),
    ("Sessions with conflict",
     len(lost_room),0,0,
     f"-{len(lost_room)}",f"-{len(lost_room)}","FCE4D6"),
    ("Capacity violations",
     ob_orig_count,ob_s1,ob_s2,
     f"-{ob_orig_count-ob_s1}",
     f"-{ob_orig_count-ob_s2}","FCE4D6"),
    ("Transitions",
     orig_transitions,trans_s1,trans_s2,
     f"-{orig_transitions-trans_s1}",
     f"-{orig_transitions-trans_s2}","EAD1F5"),
    ("ILP Parameters — First stage",
     "—",f"λ={LAMBDA}, β={BETA}","—","—","—","F2F2F2"),
    ("ILP Parameters — Second stage",
     "—","—",f"μ={MU}, β={BETA2}","—","—","F2F2F2"),
]

for i,row_data in enumerate(cmp_data,3):
    label,o,s1,s2,imp1,imp2,bg=row_data
    for col,val in enumerate([label,o,s1,s2,imp1,imp2],1):
        cell=ws_cmp.cell(row=i,column=col,value=val)
        cell.font=dfont(bold=(col==1),size=11)
        cell.fill=fl(bg); cell.alignment=center(); cell.border=brd()

ws_cmp.column_dimensions["A"].width=32
for col in ["B","C","D","E","F"]:
    ws_cmp.column_dimensions[col].width=22
ws_cmp.row_dimensions[2].height=40

# ── Save ────────────────────────────────────────────────────
out="optimization_results_final.xlsx"
wb.save(out)
print(f"\nExcel saved: {out}")

try:
    from google.colab import files
    files.download(out)
    print("Download started.")
except:
    print(f"Open '{out}' from your working directory.")

Building Excel file...

Excel saved: optimization_results_final.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started.
